# Jaxotron tour: 4D parallelism in JAX 🚀

This notebook walks through **data, FSDP, tensor (+ sequence), pipeline, and context parallelism** with [Jaxotron](https://github.com/rishiraj/jaxotron). Every command runs in two places:

- **Colab / any CPU machine**: `--cpu_devices N` simulates N accelerators, so all layouts work without a GPU.
- **Kaggle TPU v5e-8 (free)**: set the accelerator to *TPU VM v5e-8* and use the TPU section at the end.


In [ ]:
!pip install -q "jaxotron[hf,data] @ git+https://github.com/rishiraj/jaxotron"
!python -c "import jax; print('JAX', jax.__version__, '|', jax.devices())"

## 1. 8-Layout Scorecard: Single Device vs DP, FSDP, TP, TP+SP, CP, PP & 4D

`jaxotron-compare` compiles and runs one step of every parallelism strategy on 16 simulated CPU devices and verifies identical loss and gradients.

In [ ]:
!jaxotron-compare --cpu_devices 16

## 2. FSDP (ZeRO-3): every weight and AdamW moment split 8 ways

`--show_sharding` prints the `PartitionSpec` of every parameter and how many MB each device holds. The rule that does this is a single line in `jaxotron/parallel/rules.py`: `"embed": "fsdp"`.

In [ ]:
SMALL = "--seq_len 128 --global_batch_size 16 --micro_batch_size 4 --log_every 10"
!jaxotron-train --cpu_devices 8 --fsdp 8 {SMALL} --micro_batch_size 2 --dry_run --show_sharding --checkpoint_dir /tmp/tour/fsdp

## 3. Tensor + Sequence Parallelism (`--sp`) and Context Parallelism (`--cp`)

Adding `--sp` shards the residual stream over `tp`, turning `all-reduce` into `all-gather` + `reduce-scatter`.

In [ ]:
!jaxotron-train --cpu_devices 4 --tp 4 --sp {SMALL} --dry_run --show_collectives --checkpoint_dir /tmp/tour/tpsp
!jaxotron-train --cpu_devices 4 --tp 2 --cp 2 {SMALL} --steps 30 --show_collectives --checkpoint_dir /tmp/tour/tpcp

## 4. Full 4D: DP × PP × CP × TP on 16 devices

`--show_mesh` prints which `(dp, pp, cp, tp)` coordinates every device owns. The pipeline bubble is reported at startup.

In [ ]:
!jaxotron-train --cpu_devices 16 --dp 2 --pp 2 --cp 2 --tp 2 {SMALL} --steps 40 --show_mesh --show_collectives --checkpoint_dir /tmp/tour/4d

## 5. Topology-independent checkpoints + KV-cached sampling

The checkpoint above was written by 16 devices. Orbax reshards it onto a single device for fast KV-cached sampling.

In [ ]:
!jaxotron-generate --cpu --checkpoint_dir /tmp/tour/4d --prompt "5 8 11 14" --max_new_tokens 16 --temperature 0

## 6. Pretrained SmolLM2-135M and Qwen3-0.6B with Tensor Parallelism

Hugging Face safetensors stream straight into the sharded layout (no PyTorch).

In [ ]:
!jaxotron-generate --hf HuggingFaceTB/SmolLM2-135M --cpu_devices 3 --tp 3 --prompt "The three laws of motion are" --max_new_tokens 40 --temperature 0
!jaxotron-generate --hf Qwen/Qwen3-0.6B --cpu_devices 4 --tp 4 --prompt "Distributed training in JAX is" --max_new_tokens 40 --temperature 0